In [3]:
# init
import importlib, sys

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from matplotlib.markers import MarkerStyle
from matplotlib.lines import Line2D

import superconductivity.api as sc

from superconductivity.api import G0_muS
from superconductivity.api import NDArray64

from IPython import get_ipython

from tqdm import tqdm

Textwidth: float = 4.25279  # in
Textheight: float = 6.85173  # in

_ip = get_ipython()
if _ip is not None:
    _ip.run_line_magic("reload_ext", "autoreload")
    _ip.run_line_magic("autoreload", "2")

    _ip.run_line_magic(
        "config",
        "InlineBackend.print_figure_kwargs = {'bbox_inches': None, 'pad_inches': 0.0}",
    )
    _ip.run_line_magic("config", 'InlineBackend.figure_format = "retina"')  # or "png"
    _ip.run_line_magic("config", "InlineBackend.rc = {'figure.dpi': 300}")

from superconductivity.utilities.meta.axis import axis
from superconductivity.utilities.meta.param import param
from superconductivity.utilities.functions.upsampling import upsample

In [7]:
# Amplitude Study @ 7.96 GHz

Vbias0 = np.linspace(-20.0, 20.0, 3001)
Vbias = np.linspace(-4, 4, 501)
Ibias = np.linspace(-4, 4, 501)
Abias = np.linspace(0.0, 9.0, 91)

GN_G0 = 0.09230192237975306
T_K = 0.44241092170975144
Delta_meV = 0.20214864629198004
gamma_meV = 0.005524997349929867
sigmaV_mV = 0.040796080261556025
nu_GHz = 7.96
i = 21

outdir = "irradiation/nu_7.96GHz/sim.npz"

Vbias0_mV = Vbias0 * Delta_meV
Abias_mV = Abias * sc.h_pVs * nu_GHz

# actual simulation
sim = sc.sim_bcs(
    V_mV=axis("V_mV", values=Vbias0_mV),
    GN_G0=param("GN_G0", GN_G0),
    T_K=param("T_K", T_K),
    Delta_meV=param("Delta_meV", Delta_meV),
    gamma_meV=param("gamma_meV", gamma_meV),
    sigmaV_mV=param("sigmaV_mV", sigmaV_mV),
    nu_GHz=param("nu_GHz", nu_GHz),
    A_mV=axis("A_mV", values=Abias_mV),
    kernel="adaptive",
    backend="np",
)

Isim0_nA = sim["I_nA"]
Isim0 = Isim0_nA / (GN_G0 * sc.G0_muS * Delta_meV)

Vbias_mV = Vbias * Delta_meV
Abias_mV = Abias * sc.h_pVs * nu_GHz
Ibias_nA = Ibias * GN_G0 * sc.G0_muS * Delta_meV

# binning
Isim = sc.bin(
    z=upsample(Isim0, N_up=100, axis=1),
    x=upsample(Vbias0, N_up=100),
    xbins=Vbias,
    axis=1,
)
dGsim = np.gradient(Isim, Vbias, axis=1)

Vsim = sc.bin(
    z=upsample(Vbias0, N_up=100),
    x=upsample(Isim0, N_up=100, axis=1),
    xbins=Ibias,
    axis=1,
)
dRsim = np.gradient(Vsim, Ibias, axis=1)

# dGsim = savgol_filter(dGsim, window_length=10, polyorder=2, axis=1)

# saving progress
np.savez_compressed(
    outdir,
    Vbias=Vbias,
    Ibias=Ibias,
    Abias=Abias,
    Isim=Isim,
    dGsim=dGsim,
    dRsim=dRsim,
    nu_GHz=nu_GHz,
    GN_G0=GN_G0,
    Delta_meV=Delta_meV,
)